In [ ]:
#скачивание сырых ридов
name="MoPh11"
rawURL_R1="https://genedev.bionet.nsc.ru/ftp/_RawReads/2025-05-23MyGenetics/Copy%20of%20MoPh11_S86_L001_R1_001.fastq.gz"
rawURL_R2="https://genedev.bionet.nsc.ru/ftp/_RawReads/2025-05-23MyGenetics/Copy%20of%20MoPh11_S86_L001_R2_001.fastq.gz"

wget --no-check-certificate \
  -O data/raw/$name_R1.fastq.gz \
  $rawURL_R1

wget --no-check-certificate \
  -O data/raw/$name_R2.fastq.gz \
  $rawURL_R2

#проверка качества fastq
fastqc \
  data/raw/$name_R1.fastq.gz \
  data/raw/$name_R2.fastq.gz \
  -o results/fastqc_raw

#обрезка адаптеров
cutadapt \
  -q 20 \
  -m 70 \
  -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCA \
  -o data/trimmed/$name_R1.trimmed.fastq.gz \
  -p data/trimmed/$name_R2.trimmed.fastq.gz \
  data/raw/$name_R1.fastq.gz \
  data/raw/$name_R2.fastq.gz \
  > results/cutadapt/$name.cutadapt.log 2>&1

#подготовка директории Jucer 
mkdir -p data/juicer/$name/fastq
ln -sf "$(pwd)/data/trimmed/$name_R1.trimmed.fastq.gz" \
  data/juicer/$name/fastq/$name_R1.fastq.gz
ln -sf "$(pwd)/data/trimmed/$name_R2.trimmed.fastq.gz" \
  data/juicer/$name/fastq/$name_R2.fastq.gz

#запуск Jucer
bash tools/juicer/scripts/juicer.sh \
  -D "$(pwd)/tools/juicer" \
  -d "$(pwd)/data/juicer/$name" \
  -g T2T_human \
  -z "$(pwd)/data/reference/T2T_human.fna" \
  -p "$(pwd)/data/reference/chrom.sizes" \
  -y "$(pwd)/data/reference/restriction_sites_DpnII.txt" \
  -s DpnII \
  -t 32

#сохранение .his в общей папке результатов
cp data/juicer/$name/aligned/inter_30.hic results/hic/$name.inter_30.hic